---
title: "3.3 Comparing groups and small multiples"
---

<div class="lesson-tools">
<a href="https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/lessons/03-viz/03-groups-small-multiples.ipynb" class="btn btn-primary">Open in Colab</a>
<a href="https://ningzhanglab.github.io/Python-for-Beginners/downloads/03-viz/03-groups-small-multiples.ipynb" class="btn btn-outline-secondary">Download notebook</a>
</div>

**Run in your browser.** Colab needs a Google account to run and save work. If its GitHub link cannot open, download this notebook and use **File → Upload notebook** at [Colab](https://colab.research.google.com/). You can read the website without an account. A notebook is a document containing explanatory text and runnable code cells.

**Time:** about 40 minutes, including exercises · **Before you begin:** Complete [3.2 Making a chart readable](../03-viz/02-readable-charts.ipynb) first. Each notebook starts afresh: it does not remember variables from another lesson.

## You will learn

- Compare groups with both labels and distinct marker shapes.
- Use shared scales for small multiples over time.
- Explain group composition and avoid treating averages as individual relationships.

## Why it matters

A group comparison asks which observations belong together and how their patterns differ. A **small multiple** repeats the same kind of chart for different groups or periods. Repeating a chart can clarify a pattern only if the scales and sample rules remain comparable. We start with made-up numbers, then compare the same historical country panel over three observed years.

## Worked example

### Give a reader more than colour

These made-up Northland and Eastria series use the same dark colour but different markers and line patterns. A legend maps the visual cues to their names. This remains interpretable when printed in grey or viewed by someone who cannot distinguish the colours.

In [ ]:
#| label: fig-two-fictional-lines
#| fig-alt: "Two made-up life-expectancy series rise in parallel: Northland from 68 to 74 years uses circles and a solid line; Eastria from 74 to 80 uses squares and a dashed line."
#| fig-cap: "Made-up numbers; shape and line style repeat the country distinction."
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 12})
small = pd.DataFrame({"year": [2000, 2010, 2020], "Northland": [68.0, 71.0, 74.0], "Eastria": [74.0, 77.0, 80.0]})
fig_small, ax_small = plt.subplots(figsize=(6, 4), layout="constrained")
for country, marker, style in [("Northland", "o", "-"), ("Eastria", "s", "--")]:
    ax_small.plot(small["year"], small[country], label=country, marker=marker, linestyle=style, color="#21645a")
ax_small.set(xlabel="Year", ylabel="Life expectancy (years)", title="Two fictional countries · made-up numbers")
ax_small.legend()
plt.show()

The legend and marker/line differences carry the distinction even without colour. Points identify measured dates. Similar time trends in two series do not imply that one series causes the other.

### Prepare the real country panel

Run this prepared setup once in a fresh session. It uses the course CSV in a local checkout, or downloads its fixed, checksum-verified public source in Colab. A checksum is a fingerprint of a file's contents; a mismatch stops the download from being used. You do not need to retype the setup. It does not use your Google Drive or any private data.

In [ ]:
# Prepared setup: use the course CSV locally; fetch its pinned source in Colab.
from pathlib import Path
import hashlib
import io
from urllib.request import urlopen
import pandas as pd

data_file = Path("data/gapminder.csv")
if not data_file.exists():
    source = "https://raw.githubusercontent.com/jennybc/gapminder/391b5a40d3574c3b89d889dd7d72f7f3fdb4a70c/inst/extdata/gapminder.tsv"
    with urlopen(source, timeout=30) as response:
        raw = response.read()
    expected = "f74275b4a2f4a7c8e3a09a752f9d6e43d1759d78d5cbfdebaaa3e1a6efc68f78"
    if hashlib.sha256(raw).hexdigest() != expected:
        raise ValueError("The data download changed. Please report this to the course maintainer.")
    downloaded = pd.read_csv(io.BytesIO(raw), sep="\t", dtype=str)
    downloaded = downloaded.rename(columns={"continent": "region", "lifeExp": "life_exp", "pop": "population", "gdpPercap": "gdp_pc"})
    data_file.parent.mkdir(parents=True, exist_ok=True)
    downloaded.to_csv(data_file, index=False, lineterminator="\n")
print("Course data ready: data/gapminder.csv")

Look for **Course data ready**. Colab working files are temporary; rerun this cell after a runtime reset. The data cover 142 countries at five-year intervals from 1952 to 2007. Each row is one country in one year.

### Define group cues once

The source uses five broad continent groups. The dictionary below assigns a colour and marker to each. A tuple, such as `(colour, marker)`, is an ordered pair. Keeping the mapping fixed avoids making the same colour mean different groups in different panels.

In [ ]:
countries = pd.read_csv(data_file)
region_styles = {
    "Africa": ("#21645a", "o"),
    "Americas": ("#884b00", "s"),
    "Asia": ("#3d5790", "^"),
    "Europe": ("#824c8f", "D"),
    "Oceania": ("#333333", "X"),
}
print(countries.groupby("region")["country"].nunique())

Each marker means the same source group throughout this lesson. Group sizes are unequal; Oceania has only two countries in this teaching extract. The classification is not a claim that all countries within a region share a single process.

### Compare regional clouds in one cross-section

A loop draws each group onto the same Axes. Each call supplies a label, colour and marker. `legend` then displays the mapping. Keep all countries as individual points rather than plotting only regional means.

In [ ]:
#| label: fig-region-scatter
#| fig-alt: "The 2007 income\u2013life expectancy scatter uses a consistent colour and marker for each of five source regions. Groups overlap and contain substantial within-group variation."
#| fig-cap: "Historical Gapminder teaching extract via Jennifer Bryan (CC0). Country observations receive equal weight unless stated otherwise."
year_2007 = countries.loc[countries["year"] == 2007].copy()
fig_groups, ax_groups = plt.subplots(figsize=(7, 4.5), layout="constrained")
for region, (colour, marker) in region_styles.items():
    subset = year_2007.loc[year_2007["region"] == region]
    ax_groups.scatter(subset["gdp_pc"], subset["life_exp"], label=region, color=colour, marker=marker, s=32, alpha=0.85)
ax_groups.set_xscale("log")
ax_groups.set(xlabel="GDP per person (inflation-adjusted US dollars; log axis)", ylabel="Life expectancy (years)", title="Country observations by source region · 2007")
ax_groups.legend(title="Source region", loc="upper left", bbox_to_anchor=(1, 1), fontsize=9)
plt.show()

The broad pattern is visible within and across several groups, but regions overlap and individual countries differ. A slope calculated from five regional means need not equal a relationship among countries. This is an example of aggregation changing the question, not evidence about individual people's earnings or longevity.

### Repeat the chart with shared axes

`plt.subplots(3, 1)` creates three stacked Axes. `sharex=True` and `sharey=True` link their scales. Select 1957, 1982 and 2007, keep the same group cues, and derive one common income range from all three cross-sections. The vertical range is also fixed. This makes an apparent movement comparable across panels.

In [ ]:
#| label: fig-three-periods
#| fig-alt: "Three vertically stacked, shared-scale income\u2013life expectancy scatters for 1957, 1982 and 2007. Many countries move toward higher income and longer life expectancy; the same region markers and axis ranges apply throughout."
#| fig-cap: "Historical Gapminder teaching extract via Jennifer Bryan (CC0). Country observations receive equal weight unless stated otherwise. Each panel contains 142 country observations; common scales are used."
years = [1957, 1982, 2007]
selected = countries.loc[countries["year"].isin(years)]
common_limits = (selected["gdp_pc"].min() * 0.8, selected["gdp_pc"].max() * 1.2)
fig_panels, axes_panels = plt.subplots(3, 1, figsize=(6, 10), sharex=True, sharey=True, layout="constrained")
for axis, year in zip(axes_panels, years):
    current = selected.loc[selected["year"] == year]
    for region, (colour, marker) in region_styles.items():
        subset = current.loc[current["region"] == region]
        axis.scatter(subset["gdp_pc"], subset["life_exp"], label=region, color=colour, marker=marker, s=24, alpha=0.85)
    axis.set_xscale("log")
    axis.set(title=str(year), ylabel="Life expectancy (years)", xlim=common_limits, ylim=(20, 90))
axes_panels[-1].set_xlabel("GDP per person (inflation-adjusted US dollars; log axis)")
handles, labels = axes_panels[0].get_legend_handles_labels()
fig_panels.legend(handles, labels, loc="outside lower center", ncol=3, fontsize=9)
plt.show()

The panels show long-run movement and persistent dispersion, including historical exceptions. Connecting country paths or estimating within-country changes would require using country identifiers; the separate clouds do not themselves track each point through time. Look at the large figure and the alt text together; a table is preferable for exact values.

## Try it

Work in your own notebook copy. Add a new code cell for each task, try it yourself, then open the solution to compare. Run from the top if a name is missing.

### 1. Check comparability

Count distinct countries per selected year. Verify that the small-multiple x and y ranges are the same.

<details><summary>Solution 1</summary>

Run this complete solution after the worked example.

In [ ]:
print(selected.groupby("year")["country"].nunique())
print([axis.get_xlim() for axis in axes_panels])
print([axis.get_ylim() for axis in axes_panels])

Each year contains 142 countries, and the displayed ranges match. Equal counts alone would not prove the same country membership, so the outcome checks also compare country sets.

</details>

### 2. Compare two real series

Plot Canada and Japan over time using distinct markers and line styles. Preserve labels and source scope.

<details><summary>Solution 2</summary>

Run this complete solution after the worked example.

In [ ]:
#| label: fig-exercise-canada-japan
#| fig-alt: "Canada and Japan life-expectancy lines across twelve five-year observations, distinguished by circles/solid line and squares/dashed line. Japan starts lower in 1952 and ends higher in 2007."
#| fig-cap: "Historical Gapminder teaching extract via Jennifer Bryan (CC0). Country observations receive equal weight unless stated otherwise."
fig_two, ax_two = plt.subplots(figsize=(6, 4), layout="constrained")
for country, marker, style in [("Canada", "o", "-"), ("Japan", "s", "--")]:
    current = countries.loc[countries["country"] == country].sort_values("year")
    ax_two.plot(current["year"], current["life_exp"], marker=marker, linestyle=style, color="#21645a", label=country)
ax_two.set(xlabel="Observed year", ylabel="Life expectancy at birth (years)", title="Canada and Japan · historical observations")
ax_two.legend()
plt.show()

The crossing describes these historical measurements. It does not tell you which intervention produced the differing changes.

</details>

### 3. Recover the exact numbers

Create a table of Canada and Japan in 1957, 1982 and 2007 with country, year and life expectancy.

<details><summary>Solution 3</summary>

Run this complete solution after the worked example.

In [ ]:
comparison = selected.loc[selected["country"].isin(["Canada", "Japan"]), ["country", "year", "life_exp"]].sort_values(["country", "year"])
comparison

The table contains six observations. It supports exact reading alongside the overview in the plot; no visual distinction alone needs to carry every measurement.

</details>

## When it goes wrong

The following examples deliberately trigger specific errors. The prepared `try`/`except` wrapper catches that named error and prints its message so the notebook can continue. It is for these demonstrations; an unexpected error elsewhere should still stop execution. You can read the message without memorizing the wrapper.

### A requested cue is missing

A style mapping must cover every group you intend to plot.

In [ ]:
try:
    region_styles["Unspecified invented group"]
except KeyError as error:
    print(f"{type(error).__name__}: {error}")

Inspect the real group values and handle unknown labels explicitly. Do not assign them another group's colour without documenting the change.

### A non-existent subplot is indexed

There are three Axes, at positions 0, 1 and 2.

In [ ]:
try:
    axes_panels[3].set_title("Missing panel")
except IndexError as error:
    print(f"{type(error).__name__}: {error}")

The number of panels must match the supplied years. Check the list length and use zip deliberately; zip stops at the shorter collection, which can otherwise omit a requested year.

## Recap

- Keep group cues consistent and repeat distinctions with shape, line pattern or text.
- Use common scales when comparing small multiples.
- Check sample membership as well as counts.
- Regional averages, country patterns and individual relationships are different levels of analysis.

**Next:** [3.4 Saving figures for a paper or slides](../03-viz/04-saving-figures.ipynb).